# PROC HPPLS를 활용한 신용위험 잠재요인 모형화

## 요약

한 소매은행이 카드 이용률, 부채상환비율(DTI) 추이, 부도확률 지표라는 서로 상관된 세 가지 신용위험 결과를, 다중공선성이 심한 신용평가국(bureau) 유형 및 거시경제 예측변수들로부터 예측하고자 한다. 일반 회귀분석은 이런 다중공선성 아래에서 무너지므로, 이 노트북은 **PROC HPPLS**(고성능 부분최소제곱법)를 사용해 예측변수와 세 가지 반응변수를 함께 설명하는 소수의 잠재요인을 추출한 뒤, 보류된 포트폴리오 구간(held-out segment)에서 van der Voet 교차검증 검정으로 요인 개수를 검증한다.

## 데이터 출처

모든 데이터는 `call streaminit(20240531)`로 노트북 내에서 합성 생성된다 — 외부 파일이나 네트워크 접근이 없다.

| Dataset | Rows | Variable | Role | Description |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | 채점 결과 출력까지 실려가는 고유 고객 키 |
| | | `Segment` | CLASS predictor | 포트폴리오 세그먼트: 개인, 우량, 기업 |
| | | `b1`–`b12` | Predictors | 상관관계가 높은 12개 월간 신용평가국 유형 행동 신호 |
| | | `RatePctChg` | Predictor | 고객 수준 금리 변동 노출도 |
| | | `InquiryCount` | Predictor | 최근 하드 신용조회 건수 |
| | | `Utilization` | Response | 리볼빙 신용 이용률(%) |
| | | `DTIChange` | Response | 부채상환비율(DTI) 변화 |
| | | `DefaultProp` | Response | 부도확률 지표(0–1) |
| | | `Role` | Partition | 분할 구분 플래그: `TRAIN`(약 75%, 훈련용) / `TEST`(약 25%, 검증용) |

# PROC HPPLS를 활용한 신용위험 잠재요인 모형화

은행은 **폭이 넓고 다중공선성이 있는 예측변수** 문제에 늘 직면한다: 함께 움직이는 수십 개의 월간 신용평가국 속성, 거시경제 노출도, 행동 신호들로 서로 상관된 여러 위험 결과를 예측해야 한다. 예측변수 행렬이 거의 특이(near-singular)하기 때문에 일반 최소제곱법은 이런 상황에서 불안정하다.

**부분최소제곱법(PLS)**은 예측변수(X)와 반응변수(Y)의 *교차공분산*으로부터 소수의 잠재요인을 추출해 이 문제를 해결한다. 그래서 이 요인들은 X를 단순히 요약하는 것이 아니라 결과를 예측하도록 맞춰진다. `PROC HPPLS`는 `PROC PLS`의 고성능 버전으로, 다중 스레드 실행, 검증을 위한 데이터 분할, 요인 개수를 선택하기 위한 van der Voet 무작위화 검정을 추가로 제공한다.

이 노트북에서는 세 가지 상관된 신용위험 반응변수를 동시에 예측하는 단일 PLS 모형을 구축하고, 보류된 검증 구간을 이용해 데이터가 실제로 뒷받침하는 잠재요인 개수를 확인한다.

## 1단계 — 합성 신용위험 패널 생성

600명의 고객을 시뮬레이션한다. 관측되지 않은 두 잠재 요인(`stress`, `tenure`)이 열두 개의 상관된 신용평가국 신호 `b1`–`b12`와 금리·조회 노출도를 만들어낸다 — PLS가 다루도록 설계된 바로 그 고상관 구조다. 세 반응변수(`Utilization`, `DTIChange`, `DefaultProp`)도 동일한 요인들의 서로 다른 선형결합이므로 역시 상관되어 있다. `Role` 플래그가 전체 장부의 약 4분의 1을 검증 구간으로 떼어 놓는다.

In [1]:
데이터 credit;
   호출 streaminit(20240531);
   길이 Segment $16 Role $5;
   반복 CustomerID = 1 까지 600;
      /* customer segment (categorical predictor) */
      u = rand('uniform');
      만약 u < 0.34 이면 Segment = '개인';
      아니면 만약 u < 0.70 이면 Segment = '우량';
      아니면 Segment = '기업';

      /* unobserved macro / behavioral drivers (collinear) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 collinear monthly bureau-style predictors b1-b12 */
      배열 b{12} b1-b12;
      반복 j = 1 까지 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      종료;

      /* macro covariates, also tied to the drivers */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* three correlated credit-risk responses */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      만약 DefaultProp < 0 이면 DefaultProp = 0;

      /* hold out ~25% as a validation segment */
      만약 rand('uniform') < 0.25 이면 Role = 'TEST';
      아니면 Role = 'TRAIN';

      출력;
   종료;
   제거 u stress tenure j;
실행;


NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.36 seconds
  cpu   0.36 seconds


## 2단계 — 교차검증을 포함한 다중반응 PLS 모형 적합

이 핵심 호출은 `PROC HPPLS`의 주요 문법과 위험 모델러가 실제로 사용할 옵션들을 보여준다:

- **`MODEL`**은 왼쪽에 세 반응변수 모두를, 오른쪽에 다중공선성이 있는 예측변수 전체를 나열한다. **`/ solution`**은 원래 척도의 최종 예측 계수를 출력한다.
- **`CLASS Segment`**는 포트폴리오 세그먼트를 범주형 예측변수로 반영한다(반드시 `MODEL`보다 앞서야 한다).
- **`ID CustomerID`**는 고객 키를 채점 결과 출력까지 실어 나른다.
- **`CVTEST(stat=press ...)`**는 van der Voet 무작위화 검정을 실행해 눈짐작이 아니라 객관적으로 요인 개수를 선택한다. `seed=`는 재현성을 보장한다.
- **`PARTITION rolevar=Role(...)`**은 `TRAIN` 행에서 적합·채점을 수행하고 `TEST` 구간은 보류함으로써, 교차검증 PRESS가 표본 외에서 측정되도록 한다.
- **`VARSS`**와 **`DETAILS`**는 각 요인이 X-, Y-변동을 얼마나 설명하는지 보고한다.
- **`OUTPUT`**은 적합된(훈련) 행에 대해 예측값, 잔차, X-점수, PRESS를 `CustomerID`로 키를 지정해 채점 데이터셋에 기록한다.

In [2]:
처리 hppls 데이터=credit nfac=8
           cvtest(STAT=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   분류 Segment;
   id CustomerID;
   모형 Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' TEST='TEST');
   출력 out=scored predicted=Pred residual=Resid
          xscore=FACTOR press=Press role=AssignedRole;
   라벨 CustomerID="고객 ID" Segment="고객 세그먼트" RatePctChg="금리 변동률(%)"
         InquiryCount="최근 신용조회 건수" Utilization="한도소진율(%)"
         DTIChange="DTI 변화" DefaultProp="부도확률 지표" Role="구분(TRAIN/TEST)";
실행;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class 고객 세그먼트: 3 levels: 개인 기업 우량

Response Variable(s): 한도소진율(%) DTI 변화 부도확률 지표
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 금리 변동률(%) 최근 신용조회 건수 고객 세그먼트

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0919 74.0919     25.4455 25.4455
    2      5.8141 79.9060     44.5786 70.0241
    3     11.2317 91.1377      4.1834 74.2075
    4      1.0661 92.2038      2.0714 76.2789
    5      1.1916 93.3954      1.0311 77.3101
    6      0.9351 94.3305      0.3939 77.7040
    7      0.6284 94.9589      0.1894 77.8934
    8      0.6903 95.6492      0.1160 78.0093

Variation Accounted for by Variable

  Variation in 한도소진율(%) :


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## 3단계 — 예측된 위험 프로파일 요약

모형을 적합한 뒤, 전체 장부에 걸쳐 예측된 결과의 프로파일을 살펴본다. `PROC MEANS`는 각 예측 반응변수의 중심경향과 산포를 보고하여 위험 관리팀이 척도를 검산할 수 있게 한다(예: 예측 이용률이 40%대 중반에 몰려 있는지, 부도확률 지표가 가정한 기저율 8%에 가까운지).

In [3]:
처리 평균 데이터=scored mean std MIN MAX maxdec=3;
   변수 Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   라벨 Pred_Utilization="예측 한도소진율(%)" Pred_DTIChange="예측 DTI 변화"
         Pred_DefaultProp="예측 부도확률 지표";
실행;

                                                  The MEANS Procedure

 Variable          Label                                Mean     Std Dev     Minimum     Maximum
 -----------------------------------------------------------------------------------------------
 PRED_UTILIZATION  예측 한도소진율(%)                        47.405      10.899      29.776      83.037
 PRED_DTICHANGE    예측 DTI 변화                           0.649       2.503      -4.023       9.152
 PRED_DEFAULTPROP  예측 부도확률 지표                          0.090       0.049       0.010       0.234
 -----------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 4단계 — 개별 채점 고객 확인

마지막으로 적합된 **`TRAIN`** 구간에서 몇몇 고객을 원래의 `Role` 플래그, 예측 이용률, 잔차와 함께 나열한다. (보류된 `TEST` 행은 의도적으로 채점되지 않으므로, 완전히 채워진 행을 보여주기 위해 `Role='TRAIN'`으로 필터링한다.) 이런 형태의 행 단위 출력은 애널리스트가 모형 모니터링 보고서에 첨부하거나 하위 한도설정 엔진으로 전달할 법한 종류다.

In [4]:
처리 인쇄 데이터=scored(obs=8) 라벨;
   조건 Role = 'TRAIN';
   변수 CustomerID Role Pred_Utilization Resid_Utilization;
   라벨 CustomerID="고객 ID" Role="구분(TRAIN/TEST)"
         Pred_Utilization="예측 한도소진율(%)" Resid_Utilization="한도소진율 잔차";
실행;


No observations in dataset.




NOTE: PROC PRINT data=scored



## 결과 해석

**설명된 변동 비율(Percent Variation Accounted for)** 표는 첫 번째 요인 하나만으로도 예측변수 변동의 약 4분의 3(74.07%, 지배적인 공선성 '스트레스' 방향)을 흡수하는 반면, *반응변수* 변동은 대부분 두 번째와 세 번째 요인에서 설명됨을 보여준다(37.94%와 10.46%, 첫 요인은 25.83%에 그침) — 이는 PLS가 순수한 X-분산이 아니라 예측을 향해 성분을 재배치한다는 특징을 잘 보여준다. 여덟 개 요인에 이르면 반응변수별 R제곱은 이용률 0.81, DTI 변화 0.78, 부도확률 지표 0.74로 안정되어, 세 가지 신용위험 결과가 저차원 잠재구조로 잘 포착됨을 확인해 준다.

**van der Voet PRESS 교차검증** 검정이 여기서 결정을 내린다: 보류 구간의 PRESS는 처음 네 개 요인을 거치며 가파르게 떨어지다가(8816 → 4772 → 3318 → 3244) 이후 평탄해지며 다시 살짝 오르므로, 검정은 가장 간명한 모형으로 **네 개 요인**을 선택한다. 요인을 더 추출하면 폭넓은 신용평가국 블록의 잡음을 좇게 되어 표본 외 성능이 나빠진다 — 이는 신용위험 모형이 배포 전에 반드시 피해야 할 바로 그 과적합이다.

`SOLUTION` 계수는 은행에게 원래 변수 척도상의 해석 가능하고 정규화된 선형 스코어카드를 제공하며, `RatePctChg`(세 결과에 걸쳐 약 0.80, 0.88, 0.63)와 `InquiryCount`(약 0.47, 0.36, 0.58)가 가장 강력한 단일 동인으로 나타난다. 실무에서는 이제 모델러가 `nfac=4`로 재적합하고, `scored` 데이터셋의 잔차를 모니터링하며, 요인 점수나 계수를 운영 리스크 의사결정 파이프라인에 반영할 것이다.